# Adult Income Prediction

## 1. Exploratory Data Analysis (EDA)
Phần này trình bày tổng quan dữ liệu, phân phối target, và một số đặc trưng quan trọng trước khi thực hiện preprocessing.

### 1.1. Load dataset
Dataset được tải từ nguồn công khai trên GitHub để notebook có thể chạy trực tiếp trên Google Colab.

In [ ]:
# [EDA 1.1] Import module và load dataset
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import modules.eda as eda

url = "https://raw.githubusercontent.com/Hanne2202/ml-group10-data/main/adult.csv"
df = eda.load_data(url)

### 1.2. Dataset overview
Phần này kiểm tra cấu trúc tổng quát của dữ liệu, bao gồm kiểu dữ liệu, thống kê mô tả, và phân loại sơ bộ giữa các cột số và cột phân loại.

In [ ]:
# [EDA 1.2] Tổng quan cấu trúc dataset và phân loại cột
eda.dataset_overview(df)
num_cols, cat_cols = eda.get_column_types(df)

### 1.3. Missing value inspection
Trước khi xử lý dữ liệu, cần kiểm tra missing value theo chuẩn `NaN` và quan sát các giá trị bất thường trong các cột phân loại.

In [ ]:
# [EDA 1.3] Kiểm tra missing value (NaN chuẩn và ký hiệu '?')
eda.check_missing_values(df)
eda.plot_missing_summary(df)

In [ ]:
# [EDA 1.3] Kiểm tra giá trị phổ biến trong từng cột categorical
eda.inspect_categorical_values(df, cat_cols)

Kết quả cho thấy một số cột categorical xuất hiện giá trị `?`, cho thấy dữ liệu thiếu đang được mã hóa dưới dạng chuỗi thay vì `NaN`.

Vì vậy, ở bước preprocessing, nhóm sẽ cần chuẩn hóa các giá trị này về dạng missing value chuẩn trước khi thực hiện imputation.

### 1.4. Target distribution
Phần này kiểm tra phân phối của biến mục tiêu `income` để đánh giá mức độ cân bằng giữa các lớp.

In [ ]:
# [EDA 1.4] Phân phối biến mục tiêu income
eda.plot_target_distribution(df, target_col='income')

### 1.5. Categorical features by target
Phân tích một số biến phân loại tiêu biểu gồm `education`, `occupation`, và `marital-status` để quan sát sự khác biệt phân phối giữa hai nhóm thu nhập.

In [ ]:
# [EDA 1.5] Crosstab tỷ lệ và stacked bar chart cho các cột categorical tiêu biểu
# Hàm plot_categorical_by_target đã bao gồm cả print crosstab và vẽ biểu đồ
selected_cat_cols = ['education', 'occupation', 'marital-status']
eda.plot_categorical_by_target(df, selected_cat_cols, target_col='income')

Để quan sát rõ hơn sự khác biệt giữa hai nhóm thu nhập trong từng category, nhóm sử dụng bảng tỷ lệ theo hàng và biểu đồ cột chồng chuẩn hóa.

### 1.6. Numerical features by target
Phần này sử dụng boxplot để so sánh phân phối của một số biến số theo `income`.

In [ ]:
# [EDA 1.6] Boxplot các biến số tiêu biểu theo income
selected_num_cols = ['age', 'hours-per-week', 'educational-num']
eda.plot_numerical_by_target(df, selected_num_cols, target_col='income')

### 1.7. Distribution analysis for `capital-gain`
Phần này tập trung vào `capital-gain` vì biến này có phân phối lệch mạnh, tỷ lệ giá trị bằng 0 rất cao và xuất hiện một số giá trị lớn bất thường.

In [ ]:
# [EDA 1.7] Phân tích phân phối và mối liên hệ với target của capital-gain
eda.analyze_capital_feature(df, 'capital-gain', target_col='income')

Do phần lớn giá trị bằng `0`, nhóm chỉ trực quan hóa phân phối của các giá trị `capital-gain > 0` để quan sát rõ hơn phần đuôi phân phối.

### 1.8. Correlation heatmap
Phần này trực quan hóa ma trận tương quan giữa các numerical features để quan sát mức độ liên hệ tuyến tính giữa các biến số.

In [ ]:
# [EDA 1.8] Ma trận tương quan giữa các numerical features
eda.plot_correlation_heatmap(df, num_cols)

Kết quả cho thấy phần lớn numerical features chỉ có tương quan tuyến tính yếu với nhau, chưa xuất hiện cặp biến số nào có tương quan quá cao.

### 1.9. Relationship between `education` and `educational-num`
Phần này kiểm tra xem hai cột `education` và `educational-num` có đang biểu diễn cùng một thông tin hay không.

In [ ]:
# [EDA 1.9] Kiểm tra trùng lặp thông tin giữa education và educational-num
eda.check_education_redundancy(df)

Kết quả cho thấy mỗi mức của `education` tương ứng với đúng một giá trị của `educational-num`. Điều này cho thấy hai cột gần như mang cùng nội dung thông tin, chỉ khác cách biểu diễn.



### 1.10. Additional analysis of `capital-gain`
Ngoài phân phối và outlier, nhóm kiểm tra thêm mối liên hệ giữa `capital-gain` và target `income`.

In [ ]:
# [EDA 1.10] So sánh tỷ lệ capital-gain > 0 giữa hai nhóm income
import pandas as pd
import matplotlib.pyplot as plt

capital_gain_flag = pd.crosstab(
    df['income'],
    df['capital-gain'] > 0,
    normalize='index'
).round(3)
print(capital_gain_flag)

capital_gain_flag.plot(kind='bar', stacked=True, figsize=(6, 4))
plt.title('Proportion of Non-zero Capital Gain by Income')
plt.xlabel('Income')
plt.ylabel('Proportion')
plt.xticks(rotation=0)
plt.legend(title='Capital Gain > 0')
plt.tight_layout()
plt.show()

Kết quả cho thấy tỷ lệ quan sát có `capital-gain > 0` ở nhóm `>50K` cao hơn đáng kể so với nhóm `<=50K`.

Biểu đồ củng cố nhận định rằng `capital-gain` có thể là một biến hữu ích cho bài toán phân loại thu nhập, dù bản thân biến này có phân phối rất lệch.


### 1.11. Additional check for `capital-loss`
Ngoài `capital-gain`, nhóm kiểm tra thêm `capital-loss` để đánh giá xem biến này có nên bị loại bỏ ngay ở bước preprocessing hay không.

In [ ]:
# [EDA 1.11] Phân tích capital-loss trước khi quyết định có loại bỏ hay không
eda.analyze_capital_feature(df, 'capital-loss', target_col='income')

Kết quả cho thấy `capital-loss` có tỷ lệ giá trị bằng `0` rất cao, tuy nhiên tỷ lệ quan sát có giá trị khác `0` ở nhóm `>50K` vẫn cao hơn đáng kể so với nhóm `<=50K`. Vì vậy, biến này có thể vẫn mang thông tin hữu ích và chưa nên loại bỏ ngay chỉ dựa trên độ thưa của dữ liệu.